In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Imports successful.")

Imports successful.


In [2]:
PROJECT_ROOT = Path("D:/DeepFake")

DATA_ROOT = PROJECT_ROOT / "data" / "asvspoof2019"

TRAIN_DIR = DATA_ROOT / "train"
DEV_DIR = DATA_ROOT / "dev"

TRAIN_BONAFIDE_DIR = TRAIN_DIR / "bonafide"
TRAIN_SPOOF_DIR = TRAIN_DIR / "spoof"

DEV_BONAFIDE_DIR = DEV_DIR / "bonafide"
DEV_SPOOF_DIR = DEV_DIR / "spoof"

MODEL_PATH = PROJECT_ROOT / "best_voice_spoof_cnn_v2.pt"

FINAL_TRAIN_MANIFEST = (
    DATA_ROOT / "final_train_manifest.csv"
)

FINAL_DEV_MANIFEST = (
    DATA_ROOT / "final_dev_manifest.csv"
)

TEST_DIR = PROJECT_ROOT / "test_audio"

print("Project:", PROJECT_ROOT)
print("Train:", TRAIN_DIR)
print("Dev:", DEV_DIR)
print("Model:", MODEL_PATH)
print("Test audio:", TEST_DIR)

Project: D:\DeepFake
Train: D:\DeepFake\data\asvspoof2019\train
Dev: D:\DeepFake\data\asvspoof2019\dev
Model: D:\DeepFake\best_voice_spoof_cnn_v2.pt
Test audio: D:\DeepFake\test_audio


In [3]:
print("TRAIN")
print(
    "Bona-fide:",
    len(list(TRAIN_BONAFIDE_DIR.glob("*.flac")))
)
print(
    "Spoof:",
    len(list(TRAIN_SPOOF_DIR.glob("*.flac")))
)

print("\nDEV")
print(
    "Bona-fide:",
    len(list(DEV_BONAFIDE_DIR.glob("*.flac")))
)
print(
    "Spoof:",
    len(list(DEV_SPOOF_DIR.glob("*.flac")))
)

TRAIN
Bona-fide: 500
Spoof: 500

DEV
Bona-fide: 200
Spoof: 200


In [17]:
train_df = pd.read_csv(FINAL_TRAIN_MANIFEST)
dev_df = pd.read_csv(FINAL_DEV_MANIFEST)

print("Train rows:", len(train_df))
print("Dev rows:", len(dev_df))

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nDev labels:")
print(dev_df["label"].value_counts())

Train rows: 1000
Dev rows: 400

Train labels:
label
bonafide    500
spoof       500
Name: count, dtype: int64

Dev labels:
label
bonafide    200
spoof       200
Name: count, dtype: int64


In [18]:
TARGET_SR = 16000
WINDOW_SECONDS = 2
TARGET_LENGTH = TARGET_SR * WINDOW_SECONDS


def load_audio_fixed(file_path):
    audio, sr = sf.read(file_path)

    # Stereo -> mono
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    # Resample to 16 kHz
    if sr != TARGET_SR:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=TARGET_SR
        )

    # Exactly 2 seconds
    if len(audio) < TARGET_LENGTH:

        audio = np.pad(
            audio,
            (0, TARGET_LENGTH - len(audio))
        )

    elif len(audio) > TARGET_LENGTH:

        start = (
            len(audio) - TARGET_LENGTH
        ) // 2

        audio = audio[
            start:start + TARGET_LENGTH
        ]

    return audio.astype(np.float32)


def audio_to_mel(audio):

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=TARGET_SR,
        n_fft=1024,
        hop_length=256,
        n_mels=64,
        fmin=20,
        fmax=8000
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    mel_db = (
        mel_db - mel_db.mean()
    ) / (mel_db.std() + 1e-8)

    return mel_db.astype(np.float32)

print("Preprocessing ready.")

Preprocessing ready.


In [19]:
import torch.nn as nn


class VoiceSpoofCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                1, 16,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d((4, 8)),

            nn.Flatten(),

            nn.Linear(
                64 * 4 * 8,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(128, 1)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)

In [20]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

cnn_model = VoiceSpoofCNN().to(device)

cnn_model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

cnn_model.eval()

print("CNN V2 loaded.")
print("Device:", device)

CNN V2 loaded.
Device: cpu


TESTING CNN

In [79]:
from pathlib import Path

cloned_voice_path = Path(
    r"D:\DeepFake\test_audio\Y_real.wav"
)

print("File exists:", cloned_voice_path.exists())

File exists: True


In [80]:
audio = load_audio_fixed(cloned_voice_path)

mel = audio_to_mel(audio)

x = torch.tensor(
    mel,
    dtype=torch.float32
).unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    logits = cnn_model(x)
    spoof_probability = torch.sigmoid(logits).item()

prediction = (
    "SPOOF"
    if spoof_probability >= 0.5
    else "BONAFIDE"
)

print("File:", cloned_voice_path.name)
print("Prediction:", prediction)
print("Spoof probability:", round(spoof_probability, 4))
print("Bonafide probability:", round(1 - spoof_probability, 4))

File: Y_real.wav
Prediction: SPOOF
Spoof probability: 0.9422
Bonafide probability: 0.0578


In [8]:
example_file = TRAIN_BONAFIDE_DIR.glob("*.flac")
example_file = next(example_file)

audio = load_audio_fixed(example_file)

mel = audio_to_mel(audio)

x = torch.tensor(
    mel,
    dtype=torch.float32
).unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():

    logits = cnn_model(x)
    cnn_score = torch.sigmoid(logits).item()

print("File:", example_file.name)
print("Audio shape:", audio.shape)
print("Mel shape:", mel.shape)
print("CNN spoof score:", round(cnn_score, 4))

d:\DeepFake\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


File: LA_T_1013597.flac
Audio shape: (32000,)
Mel shape: (64, 126)
CNN spoof score: 0.0339


DETECTOR

In [9]:
TARGET_SR = 16000
WINDOW_SECONDS = 2
TARGET_LENGTH = TARGET_SR * WINDOW_SECONDS

print("Target sample rate:", TARGET_SR)
print("Window:", WINDOW_SECONDS, "seconds")
print("Target samples:", TARGET_LENGTH)

Target sample rate: 16000
Window: 2 seconds
Target samples: 32000


In [10]:
import numpy as np
import soundfile as sf
import librosa


def load_audio_fixed(file_path):
    audio, sr = sf.read(file_path)

    # Stereo -> mono
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    # Resample
    if sr != TARGET_SR:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=TARGET_SR
        )

    # Center crop / pad to exactly 2 seconds
    if len(audio) < TARGET_LENGTH:
        audio = np.pad(
            audio,
            (0, TARGET_LENGTH - len(audio))
        )

    elif len(audio) > TARGET_LENGTH:
        start = (len(audio) - TARGET_LENGTH) // 2
        audio = audio[
            start:start + TARGET_LENGTH
        ]

    return audio.astype(np.float32)


print("Audio loader ready.")

Audio loader ready.


In [11]:
def extract_basic_features(audio, sr=TARGET_SR):
    
    # -----------------------------
    # Frame-level energy
    # -----------------------------
    rms = librosa.feature.rms(
        y=audio,
        frame_length=1024,
        hop_length=256
    )[0]

    # -----------------------------
    # Zero crossing rate
    # -----------------------------
    zcr = librosa.feature.zero_crossing_rate(
        audio,
        frame_length=1024,
        hop_length=256
    )[0]

    # -----------------------------
    # Spectral features
    # -----------------------------
    centroid = librosa.feature.spectral_centroid(
        y=audio,
        sr=sr,
        n_fft=1024,
        hop_length=256
    )[0]

    bandwidth = librosa.feature.spectral_bandwidth(
        y=audio,
        sr=sr,
        n_fft=1024,
        hop_length=256
    )[0]

    rolloff = librosa.feature.spectral_rolloff(
        y=audio,
        sr=sr,
        n_fft=1024,
        hop_length=256
    )[0]

    flatness = librosa.feature.spectral_flatness(
        y=audio,
        n_fft=1024,
        hop_length=256
    )[0]

    # -----------------------------
    # Silence / pause estimate
    # -----------------------------
    energy_threshold = 0.10 * np.max(rms)

    silent_frames = rms < energy_threshold

    pause_ratio = np.mean(silent_frames)

    # -----------------------------
    # Energy dynamics
    # -----------------------------
    energy_mean = np.mean(rms)
    energy_std = np.std(rms)

    # -----------------------------
    # Aggregate features
    # -----------------------------
    features = {
        "energy_mean": energy_mean,
        "energy_std": energy_std,

        "zcr_mean": np.mean(zcr),
        "zcr_std": np.std(zcr),

        "spectral_centroid_mean": np.mean(centroid),
        "spectral_centroid_std": np.std(centroid),

        "spectral_bandwidth_mean": np.mean(bandwidth),
        "spectral_bandwidth_std": np.std(bandwidth),

        "spectral_rolloff_mean": np.mean(rolloff),
        "spectral_rolloff_std": np.std(rolloff),

        "spectral_flatness_mean": np.mean(flatness),
        "spectral_flatness_std": np.std(flatness),

        "pause_ratio": pause_ratio
    }

    return features

In [12]:
example_file = TRAIN_BONAFIDE_DIR / "LA_T_1013597.flac"

audio = load_audio_fixed(example_file)

features = extract_basic_features(
    audio,
    TARGET_SR
)

print("File:", example_file.name)

for name, value in features.items():
    print(f"{name}: {value:.6f}")

File: LA_T_1013597.flac
energy_mean: 0.124022
energy_std: 0.081797
zcr_mean: 0.151080
zcr_std: 0.173405
spectral_centroid_mean: 1828.858082
spectral_centroid_std: 1509.869998
spectral_bandwidth_mean: 1568.156764
spectral_bandwidth_std: 593.892101
spectral_rolloff_mean: 3242.063492
spectral_rolloff_std: 2255.249355
spectral_flatness_mean: 0.037502
spectral_flatness_std: 0.068248
pause_ratio: 0.158730


In [13]:
def extract_pitch_features(audio, sr=TARGET_SR):

    f0, voiced_flag, voiced_prob = librosa.pyin(
        audio,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr,
        frame_length=1024,
        hop_length=256
    )

    valid_f0 = f0[~np.isnan(f0)]

    if len(valid_f0) == 0:
        return {
            "f0_mean": 0.0,
            "f0_std": 0.0,
            "f0_min": 0.0,
            "f0_max": 0.0,
            "f0_range": 0.0,
            "voiced_ratio": 0.0
        }

    return {
        "f0_mean": np.mean(valid_f0),
        "f0_std": np.std(valid_f0),
        "f0_min": np.min(valid_f0),
        "f0_max": np.max(valid_f0),
        "f0_range": np.max(valid_f0) - np.min(valid_f0),
        "voiced_ratio": np.mean(voiced_flag)
    }

In [14]:
pitch_features = extract_pitch_features(
    audio,
    TARGET_SR
)

for name, value in pitch_features.items():
    print(f"{name}: {value:.6f}")

f0_mean: 185.782719
f0_std: 10.113750
f0_min: 166.728822
f0_max: 207.652349
f0_range: 40.923526
voiced_ratio: 0.849206


In [15]:
def extract_naturalness_features(file_path):

    audio = load_audio_fixed(file_path)

    basic = extract_basic_features(
        audio,
        TARGET_SR
    )

    pitch = extract_pitch_features(
        audio,
        TARGET_SR
    )

    features = {
        **basic,
        **pitch
    }

    return features

In [16]:
all_features = extract_naturalness_features(
    example_file
)

print("Number of features:", len(all_features))

for name, value in all_features.items():
    print(f"{name}: {value:.6f}")

Number of features: 19
energy_mean: 0.124022
energy_std: 0.081797
zcr_mean: 0.151080
zcr_std: 0.173405
spectral_centroid_mean: 1828.858082
spectral_centroid_std: 1509.869998
spectral_bandwidth_mean: 1568.156764
spectral_bandwidth_std: 593.892101
spectral_rolloff_mean: 3242.063492
spectral_rolloff_std: 2255.249355
spectral_flatness_mean: 0.037502
spectral_flatness_std: 0.068248
pause_ratio: 0.158730
f0_mean: 185.782719
f0_std: 10.113750
f0_min: 166.728822
f0_max: 207.652349
f0_range: 40.923526
voiced_ratio: 0.849206


In [17]:
from tqdm.auto import tqdm

train_feature_rows = []

for _, row in tqdm(
    train_df.iterrows(),
    total=len(train_df),
    desc="Extracting TRAIN features"
):

    file_path = Path(row["path"])

    features = extract_naturalness_features(
        file_path
    )

    features["audio_id"] = row["audio_id"]
    features["label"] = row["label"]
    features["label_id"] = row["label_id"]

    train_feature_rows.append(features)

train_features_df = pd.DataFrame(
    train_feature_rows
)

print(
    "TRAIN feature table shape:",
    train_features_df.shape
)

print(train_features_df.head())

Extracting TRAIN features: 100%|██████████| 1000/1000 [04:05<00:00,  4.07it/s]

TRAIN feature table shape: (1000, 22)
   energy_mean  energy_std  zcr_mean   zcr_std  spectral_centroid_mean  \
0     0.124022    0.081797  0.151080  0.173405             1828.858082   
1     0.035432    0.062705  0.139958  0.179731             1722.021306   
2     0.090623    0.080996  0.111406  0.108851             1675.079160   
3     0.110094    0.098952  0.189391  0.161768             2164.839391   
4     0.078707    0.085832  0.131913  0.145035             1924.015437   

   spectral_centroid_std  spectral_bandwidth_mean  spectral_bandwidth_std  \
0            1509.869998              1568.156764              593.892101   
1            1459.504606              1373.744713              800.090819   
2            1152.002343              1431.793738              482.662218   
3            1229.798489              1563.696710              345.689578   
4            1196.633357              1788.328055              472.804715   

   spectral_rolloff_mean  spectral_rolloff_std  ...  p

In [19]:
print("train_features_df exists:",
      "train_features_df" in globals())

print("dev_features_df exists:",
      "dev_features_df" in globals())

print("\nFeature-related variables:")

for name in globals():
    if "feature" in name.lower():
        print(name)

train_features_df exists: True
dev_features_df exists: False

Feature-related variables:
extract_basic_features
features
extract_pitch_features
pitch_features
extract_naturalness_features
all_features
train_feature_rows
train_features_df


In [20]:
from pathlib import Path

DEV_FEATURE_PATH = Path(
    "D:/DeepFake/data/asvspoof2019/dev_naturalness_features.csv"
)

print("DEV feature file exists:", DEV_FEATURE_PATH.exists())

DEV feature file exists: False


In [21]:
from tqdm.auto import tqdm

dev_feature_rows = []

for _, row in tqdm(
    dev_df.iterrows(),
    total=len(dev_df),
    desc="Extracting DEV features"
):

    file_path = Path(row["path"])

    features = extract_naturalness_features(
        file_path
    )

    features["audio_id"] = row["audio_id"]
    features["label"] = row["label"]
    features["label_id"] = row["label_id"]

    dev_feature_rows.append(features)

dev_features_df = pd.DataFrame(
    dev_feature_rows
)

print("DEV feature table shape:",
      dev_features_df.shape)

Extracting DEV features: 100%|██████████| 400/400 [02:18<00:00,  2.89it/s]

DEV feature table shape: (400, 22)


In [22]:
DEV_FEATURE_PATH = Path(
    "D:/DeepFake/data/asvspoof2019/dev_naturalness_features.csv"
)

dev_features_df.to_csv(
    DEV_FEATURE_PATH,
    index=False
)

print("Saved:", DEV_FEATURE_PATH)

Saved: D:\DeepFake\data\asvspoof2019\dev_naturalness_features.csv


In [23]:
print("TRAIN:", train_features_df.shape)
print("DEV:", dev_features_df.shape)

print("\nTRAIN labels:")
print(train_features_df["label"].value_counts())

print("\nDEV labels:")
print(dev_features_df["label"].value_counts())

TRAIN: (1000, 22)
DEV: (400, 22)

TRAIN labels:
label
bonafide    500
spoof       500
Name: count, dtype: int64

DEV labels:
label
bonafide    200
spoof       200
Name: count, dtype: int64


In [24]:
print("TRAIN shape:", train_features_df.shape)
print("DEV shape:", dev_features_df.shape)

print("\nTRAIN labels:")
print(train_features_df["label"].value_counts())

print("\nDEV labels:")
print(dev_features_df["label"].value_counts())

print("\nMissing values:")
print(
    "TRAIN:",
    train_features_df.isna().sum().sum()
)

print(
    "DEV:",
    dev_features_df.isna().sum().sum()
)

TRAIN shape: (1000, 22)
DEV shape: (400, 22)

TRAIN labels:
label
bonafide    500
spoof       500
Name: count, dtype: int64

DEV labels:
label
bonafide    200
spoof       200
Name: count, dtype: int64

Missing values:
TRAIN: 0
DEV: 0


In [25]:
feature_columns = [
    c for c in train_features_df.columns
    if c not in [
        "audio_id",
        "label",
        "label_id"
    ]
]

X_train = train_features_df[feature_columns].copy()
y_train = train_features_df["label_id"].copy()

X_dev = dev_features_df[feature_columns].copy()
y_dev = dev_features_df["label_id"].copy()

print("Number of features:", len(feature_columns))
print("X_train:", X_train.shape)
print("X_dev:", X_dev.shape)

Number of features: 19
X_train: (1000, 19)
X_dev: (400, 19)


In [26]:
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_dev = X_dev.fillna(train_medians)

print("Missing values after filling:")
print("TRAIN:", X_train.isna().sum().sum())
print("DEV:", X_dev.isna().sum().sum())

Missing values after filling:
TRAIN: 0
DEV: 0


In [27]:
from sklearn.ensemble import RandomForestClassifier

naturalness_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

naturalness_model.fit(
    X_train,
    y_train
)

print("Naturalness-only model trained.")

Naturalness-only model trained.


In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

dev_naturalness_scores = (
    naturalness_model.predict_proba(X_dev)[:, 1]
)

dev_naturalness_pred = (
    dev_naturalness_scores >= 0.5
).astype(int)

print(
    "Accuracy:",
    accuracy_score(y_dev, dev_naturalness_pred)
)

print(
    "Precision:",
    precision_score(
        y_dev,
        dev_naturalness_pred,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_dev,
        dev_naturalness_pred,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_dev,
        dev_naturalness_pred,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_dev,
        dev_naturalness_scores
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_dev,
        dev_naturalness_pred
    )
)

Accuracy: 0.7475
Precision: 0.746268656716418
Recall: 0.75
F1: 0.7481296758104738
ROC-AUC: 0.814775

Confusion Matrix:
[[149  51]
 [ 50 150]]


AURIGIN API KEY

In [5]:
from dotenv import load_dotenv
import os

load_dotenv("D:/DeepFake/.env")

AURIGIN_API_KEY = os.getenv("AURIGIN_API_KEY")

if not AURIGIN_API_KEY:
    raise RuntimeError(
        "AURIGIN_API_KEY not found in D:/DeepFake/.env"
    )

print("Aurigin API key loaded:", True)

Aurigin API key loaded: True


In [36]:
from pathlib import Path

phone_voice_path = Path(
    "D:/DeepFake/test_audio/phone_real_voice.wav"
)

print("Exists:", phone_voice_path.exists())

if phone_voice_path.exists():
    print("Size:", phone_voice_path.stat().st_size, "bytes")

Exists: True
Size: 368718 bytes


In [38]:
import requests

url = "https://api.aurigin.ai/v1/predict"

headers = {
    "x-api-key": AURIGIN_API_KEY
}

with open(phone_voice_path, "rb") as audio_file:

    files = {
        "file": (
            phone_voice_path.name,
            audio_file,
            "audio/wav"
        )
    }

    response = requests.post(
        url,
        headers=headers,
        files=files,
        timeout=120
    )

print("HTTP status:", response.status_code)
print(response.text)

HTTP status: 200
{"prediction_id":"pred_d86c1dbfca79","global":{"confidence":0.9403497775395712,"result":"bonafide","reason":null,"score":0.029869861072964136},"segments":[{"start":0.0,"end":11.52,"confidence":0.9403497775395712,"result":"bonafide","scores":[0.028801772743463516,0.03099658712744713,0.029676973819732666]}],"model":"apollo-5-1-2026-07-20","processing_time":0.8427622318267822,"audio_duration":11.52,"warnings":[]}


In [39]:
aurigin_result = response.json()

print(aurigin_result)

{'prediction_id': 'pred_d86c1dbfca79', 'global': {'confidence': 0.9403497775395712, 'result': 'bonafide', 'reason': None, 'score': 0.029869861072964136}, 'segments': [{'start': 0.0, 'end': 11.52, 'confidence': 0.9403497775395712, 'result': 'bonafide', 'scores': [0.028801772743463516, 0.03099658712744713, 0.029676973819732666]}], 'model': 'apollo-5-1-2026-07-20', 'processing_time': 0.8427622318267822, 'audio_duration': 11.52, 'warnings': []}


In [40]:
global_result = aurigin_result.get("global", {})

print("Global result:",
      global_result.get("result"))

print("Global confidence:",
      global_result.get("confidence"))

Global result: bonafide
Global confidence: 0.9403497775395712


In [41]:
for segment in aurigin_result.get("segments", []):
    print(
        f"{segment['start']:.1f}s - "
        f"{segment['end']:.1f}s : "
        f"{segment['result']} "
        f"(confidence={segment['confidence']:.4f})"
    )

0.0s - 11.5s : bonafide (confidence=0.9403)


READY FOR USING THE AURIGIN API KEY

In [42]:
from pathlib import Path
import soundfile as sf

# Pick one known spoof file from our ASVspoof training set
known_spoof_file = next(
    TRAIN_SPOOF_DIR.glob("*.flac")
)

audio, sr = sf.read(known_spoof_file)

duration = len(audio) / sr

print("File:", known_spoof_file.name)
print("Sample rate:", sr)
print("Duration:", round(duration, 3), "seconds")
print("True label: SPOOF")

File: LA_T_1026130.flac
Sample rate: 16000
Duration: 3.577 seconds
True label: SPOOF


In [43]:
def aurigin_detect(file_path):

    file_path = Path(file_path)

    url = "https://api.aurigin.ai/v1/predict"

    headers = {
        "x-api-key": AURIGIN_API_KEY
    }

    with open(file_path, "rb") as audio_file:

        files = {
            "file": (
                file_path.name,
                audio_file,
                "audio/flac"
            )
        }

        response = requests.post(
            url,
            headers=headers,
            files=files,
            timeout=120
        )

    response.raise_for_status()

    return response.json()

In [45]:
spoof_api_result = aurigin_detect(
    known_spoof_file
)

print(spoof_api_result)

{'prediction_id': 'pred_abedcc5995f6', 'global': {'confidence': 0.9356545209884644, 'result': 'spoofed', 'reason': None, 'score': 0.9678272604942322}, 'segments': [{'start': 0.0, 'end': 3.5770625, 'confidence': 0.9356545209884644, 'result': 'spoofed', 'scores': [0.9678272604942322]}], 'model': 'apollo-5-1-2026-07-20', 'processing_time': 0.5776090621948242, 'audio_duration': 3.58, 'warnings': []}


WILL TEST THIS  TRAINING USING REAL CALL RECORDING

In [6]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

AURIGIN_API_KEY = os.getenv("AURIGIN_API_KEY")

if not AURIGIN_API_KEY:
    raise ValueError("AURIGIN_API_KEY was not found in your .env file")

AURIGIN_URL = "https://api.aurigin.ai/v1/predict"


def classify_with_aurigin(audio_path):

    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"File not found: {audio_path}")

    print("Sending:", audio_path)

    with open(audio_path, "rb") as audio_file:

        response = requests.post(
            AURIGIN_URL,
            headers={
                "x-api-key": AURIGIN_API_KEY
            },
            files={
                "file": (
                    os.path.basename(audio_path),
                    audio_file,
                    "audio/wav"
                )
            },
            timeout=120
        )

    print("HTTP status:", response.status_code)
    print("Response:", response.text)

    if response.status_code != 200:
        return None

    result = response.json()

    global_result = result.get("global", {})

    print("\nPrediction ID:", result.get("prediction_id"))
    print("Result:", global_result.get("result"))
    print("Confidence:", global_result.get("confidence"))
    print("Score:", global_result.get("score"))

    return result

In [53]:
result_real = classify_with_aurigin(
    r"D:\DeepFake\test_audio\phone_real_voice.wav"
)

Sending: D:\DeepFake\test_audio\phone_real_voice.wav
HTTP status: 200
Response: {"prediction_id":"pred_74c128364439","global":{"confidence":0.9403497775395712,"result":"bonafide","reason":null,"score":0.029869861072964136},"segments":[{"start":0.0,"end":11.52,"confidence":0.9403497775395712,"result":"bonafide","scores":[0.028801772743463516,0.03099658712744713,0.029676973819732666]}],"model":"apollo-5-1-2026-07-20","processing_time":0.8073198795318604,"audio_duration":11.52,"warnings":[]}

Prediction ID: pred_74c128364439
Result: bonafide
Confidence: 0.9403497775395712
Score: 0.029869861072964136


In [54]:
BONAFIDE_DIR = r"D:\DeepFake\data\asvspoof2019\dev\bonafide"
SPOOF_DIR = r"D:\DeepFake\data\asvspoof2019\dev\spoof"

CLASSIFYING

In [55]:
import os
import pandas as pd
import time

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

BONAFIDE_DIR = r"D:\DeepFake\data\asvspoof2019\dev\bonafide"
SPOOF_DIR = r"D:\DeepFake\data\asvspoof2019\dev\spoof"

OUTPUT_CSV = r"D:\DeepFake\aurigin_classification_results.csv"

N_BONAFIDE = 20
N_SPOOF = 20


# --------------------------------------------------
# COLLECT AUDIO FILES
# --------------------------------------------------

audio_extensions = {".wav", ".flac", ".mp3", ".m4a"}

bonafide_files = sorted([
    os.path.join(BONAFIDE_DIR, f)
    for f in os.listdir(BONAFIDE_DIR)
    if os.path.splitext(f)[1].lower() in audio_extensions
])

spoof_files = sorted([
    os.path.join(SPOOF_DIR, f)
    for f in os.listdir(SPOOF_DIR)
    if os.path.splitext(f)[1].lower() in audio_extensions
])


print("Available bona-fide files:", len(bonafide_files))
print("Available spoof files:", len(spoof_files))


# --------------------------------------------------
# SELECT ONLY 20 + 20
# --------------------------------------------------

selected_files = []

for path in bonafide_files[:N_BONAFIDE]:
    selected_files.append((path, "bonafide"))

for path in spoof_files[:N_SPOOF]:
    selected_files.append((path, "spoof"))


print("\nFiles selected for Aurigin:")
print("Bona-fide:", N_BONAFIDE)
print("Spoof:", N_SPOOF)
print("Total:", len(selected_files))


# --------------------------------------------------
# LOAD EXISTING RESULTS
# --------------------------------------------------

if os.path.exists(OUTPUT_CSV):
    results_df = pd.read_csv(OUTPUT_CSV)
    print("\nExisting results found:", len(results_df))
else:
    results_df = pd.DataFrame(columns=[
        "file",
        "dataset_label",
        "aurigin_result",
        "aurigin_confidence",
        "aurigin_score",
        "prediction_id"
    ])
    print("\nNo previous results found. Starting fresh.")


# --------------------------------------------------
# CLASSIFY
# --------------------------------------------------

for audio_path, true_label in selected_files:

    filename = os.path.basename(audio_path)

    # Prevent duplicate API calls
    if filename in results_df["file"].values:
        print(f"\nSKIPPING (already classified): {filename}")
        continue

    print("\n" + "=" * 60)
    print("Classifying:", filename)
    print("Dataset label:", true_label)

    try:
        result = classify_with_aurigin(audio_path)

        if result is None:
            print("No result returned.")
            continue

        global_result = result.get("global", {})

        aurigin_result = global_result.get("result")
        confidence = global_result.get("confidence")
        score = global_result.get("score")
        prediction_id = result.get("prediction_id")

        new_row = {
            "file": filename,
            "dataset_label": true_label,
            "aurigin_result": aurigin_result,
            "aurigin_confidence": confidence,
            "aurigin_score": score,
            "prediction_id": prediction_id
        }

        # Add result immediately
        results_df = pd.concat(
            [results_df, pd.DataFrame([new_row])],
            ignore_index=True
        )

        # Save immediately
        results_df.to_csv(OUTPUT_CSV, index=False)

        print("\nSaved:")
        print(new_row)

    except Exception as e:
        print("\nERROR:", e)
        print("Moving to the next file.")

    # Small delay between requests
    time.sleep(1)


print("\n" + "=" * 60)
print("BATCH COMPLETE")

print("\nResults saved to:")
print(OUTPUT_CSV)

print("\nTotal saved classifications:", len(results_df))

display(results_df)

Available bona-fide files: 200
Available spoof files: 200

Files selected for Aurigin:
Bona-fide: 20
Spoof: 20
Total: 40

No previous results found. Starting fresh.

Classifying: LA_D_1069525.flac
Dataset label: bonafide
Sending: D:\DeepFake\data\asvspoof2019\dev\bonafide\LA_D_1069525.flac
HTTP status: 200
Response: {"prediction_id":"pred_5b7e976a98a2","global":{"confidence":0.9523849952965975,"result":"bonafide","reason":null,"score":0.023961542063164126},"segments":[{"start":0.0,"end":5.1155,"confidence":0.9523849952965975,"result":"bonafide","scores":[0.023968826979398727,0.023646177724003792]}],"model":"apollo-5-1-2026-07-20","processing_time":0.6829404830932617,"audio_duration":5.12,"warnings":[]}

Prediction ID: pred_5b7e976a98a2
Result: bonafide
Confidence: 0.9523849952965975
Score: 0.023961542063164126

Saved:
{'file': 'LA_D_1069525.flac', 'dataset_label': 'bonafide', 'aurigin_result': 'bonafide', 'aurigin_confidence': 0.9523849952965975, 'aurigin_score': 0.023961542063164126, 

,file,dataset_label,aurigin_result,aurigin_confidence,aurigin_score,prediction_id
0,LA_D_1069525.flac,bonafide,bonafide,0.952385,0.023962,pred_5b7e976a98a2
1,LA_D_1107267.flac,bonafide,bonafide,0.941576,0.029212,pred_52491544c0c0
2,LA_D_1125976.flac,bonafide,bonafide,0.950879,0.02456,pred_28b46e08e149
3,LA_D_1299580.flac,bonafide,bonafide,0.947146,0.026427,pred_e7c638c5a7f4
4,LA_D_1316596.flac,bonafide,bonafide,0.952623,0.023688,pred_cea5e90a7943
5,LA_D_1330775.flac,bonafide,bonafide,0.94623,0.026885,pred_e6dfcbcfb881
6,LA_D_1424876.flac,bonafide,bonafide,0.934731,0.032635,pred_325574ee012f
7,LA_D_1475288.flac,bonafide,bonafide,0.950479,0.024761,pred_3ab4aeed6652
8,LA_D_1511694.flac,bonafide,bonafide,0.942289,0.028855,pred_a47ac6963a14
9,LA_D_1702465.flac,bonafide,bonafide,0.941002,0.029499,pred_a81f53565d77


In [56]:
import pandas as pd

csv_path = r"D:\DeepFake\aurigin_classification_results.csv"

aurigin_df = pd.read_csv(csv_path)

print("Total classifications:", len(aurigin_df))

print("\nDataset labels:")
print(aurigin_df["dataset_label"].value_counts())

print("\nAurigin predictions:")
print(aurigin_df["aurigin_result"].value_counts())

Total classifications: 40

Dataset labels:
dataset_label
bonafide    20
spoof       20
Name: count, dtype: int64

Aurigin predictions:
aurigin_result
bonafide    20
spoofed     20
Name: count, dtype: int64


In [57]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Convert labels to binary
y_true = aurigin_df["dataset_label"].map({
    "bonafide": 0,
    "spoof": 1
})

y_pred = aurigin_df["aurigin_result"].map({
    "bonafide": 0,
    "spoofed": 1
})

print("Accuracy:",
      accuracy_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["bonafide", "spoof"]
    )
)

Accuracy: 1.0

Confusion Matrix:
[[20  0]
 [ 0 20]]

Classification Report:
              precision    recall  f1-score   support

    bonafide       1.00      1.00      1.00        20
       spoof       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [58]:
errors = aurigin_df[y_true != y_pred]

print("Number of errors:", len(errors))

display(
    errors[
        [
            "file",
            "dataset_label",
            "aurigin_result",
            "aurigin_confidence",
            "aurigin_score"
        ]
    ]
)

Number of errors: 0


,file,dataset_label,aurigin_result,aurigin_confidence,aurigin_score


In [15]:
import soundfile as sf
import numpy as np
import os

call_path = r"D:\DeepFake\test_audio\Y_test_3.wav"

print("File exists:", os.path.exists(call_path))

audio, sr = sf.read(call_path)

print("Sample rate:", sr)
print("Channels/shape:", audio.shape)
print("Duration:", round(len(audio) / sr, 2), "seconds")
print("Min:", audio.min())
print("Max:", audio.max())
print("RMS:", np.sqrt(np.mean(audio.astype(float) ** 2)))

File exists: True
Sample rate: 44100
Channels/shape: (288000,)
Duration: 6.53 seconds
Min: -0.906402587890625
Max: 0.911956787109375
RMS: 0.17728030770423275


In [16]:
result_call = classify_with_aurigin(
    call_path
)

Sending: D:\DeepFake\test_audio\Y_test_3.wav
HTTP status: 403
Response: {"detail":"No credits remaining. Please upgrade your plan."}
